In [1]:
# ============================================================
# [목적]
#   hourly aggregation 데이터에서 train만 증강
#   - 원본 duration이 300~3600초 사이로 다양함
#   - train을 2배, 3배... 12배까지 복제 (단, 60분=3600초 초과 안되게 cap)
#   - test는 원본 그대로 (실제 외삽 성능 측정)
# ============================================================

import os
import yaml
import pickle
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import gc
import json

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
CONFIG_PATH = "/content/config.yaml"
with open(CONFIG_PATH, 'r') as f:
    config = yaml.safe_load(f)

DRIVE_ROOT     = f"/content/drive/MyDrive/{config['project_name']}"
PROCESSED_PATH = os.path.join(DRIVE_ROOT, config['paths']['processed_data'])
MODEL_PATH     = os.path.join(DRIVE_ROOT, config['paths']['models'])
REPORT_PATH    = os.path.join(DRIVE_ROOT, config['paths']['outputs'], "reports")

In [4]:
# ============================================================
# 데이터 로드
# ============================================================
hourly_file = os.path.join(PROCESSED_PATH, "instance_usage_hourly_processed.parquet")
df = pd.read_parquet(hourly_file)
print(f"Hourly rows: {len(df):,}")
print(f"\n[duration 통계]")
print(df['duration'].describe())

# ============================================================
# 피처 / 타겟
# ============================================================
features = ["cpu", "memory", "duration", "hour"]
target   = "energy_kwh"

X = df[features]
y = df[target]

Hourly rows: 1,454,606

[duration 통계]
count    1.454606e+06
mean     3.265929e+03
std      1.804283e+03
min      1.000000e+00
25%      3.065000e+03
50%      3.600000e+03
75%      3.600000e+03
max      2.353400e+04
Name: duration, dtype: float64


In [5]:
# ============================================================
# Train / Test 분할 (test는 원본 그대로!)
# ============================================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=config['model']['test_size'],
    random_state=config['model']['random_state']
)
print(f"\nTrain (원본): {len(X_train):,}")
print(f"Test  (원본): {len(X_test):,}")

# ============================================================
# Train만 증강
# duration * multiplier 가 3600초 초과하지 않도록 cap
# energy_kwh도 동일 비율로 증가 (선형 관계)
# ============================================================
MAX_DURATION = 3600.0   # 60분 = 3600초

augmented = [pd.concat([X_train, y_train.rename("energy_kwh")], axis=1)]

for multiplier in [2, 3, 4, 6, 8, 12]:
    df_aug = pd.concat([X_train, y_train.rename("energy_kwh")], axis=1).copy()

    # duration * multiplier가 MAX_DURATION 초과하는 행은 MAX_DURATION으로 cap
    new_duration = (df_aug['duration'] * multiplier).clip(upper=MAX_DURATION)
    actual_mult = new_duration / df_aug['duration']  # 실제 배율 (cap 때문에 다를 수 있음)

    df_aug['duration'] = new_duration
    df_aug['energy_kwh'] = df_aug['energy_kwh'] * actual_mult

    augmented.append(df_aug)

df_train_aug = pd.concat(augmented, ignore_index=True)
del augmented; gc.collect()

X_train_aug = df_train_aug[features]
y_train_aug = df_train_aug[target]
del df_train_aug; gc.collect()

print(f"Train (증강): {len(X_train_aug):,}")
print(f"\n[증강 후 duration 통계]")
print(X_train_aug['duration'].describe())


Train (원본): 1,163,684
Test  (원본): 290,922
Train (증강): 8,145,788

[증강 후 duration 통계]
count    8.145788e+06
mean     3.269644e+03
std      1.080096e+03
min      1.000000e+00
25%      3.600000e+03
50%      3.600000e+03
75%      3.600000e+03
max      2.353400e+04
Name: duration, dtype: float64


In [6]:
# ============================================================
# RF 학습
# ============================================================
rf = RandomForestRegressor(
    n_estimators=100,
    max_depth=15,
    min_samples_split=10,
    min_samples_leaf=5,
    random_state=config['model']['random_state'],
    n_jobs=-1
)
rf.fit(X_train_aug, y_train_aug)
print(f"\nRF trained!")
print(f"feature_names_in_: {rf.feature_names_in_}")


RF trained!
feature_names_in_: ['cpu' 'memory' 'duration' 'hour']


In [7]:
# ============================================================
# 평가 (test는 원본)
# ============================================================
y_pred = rf.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae  = mean_absolute_error(y_test, y_pred)
r2   = r2_score(y_test, y_pred)
mape = np.mean(np.abs((y_test.values - y_pred) / y_test.values)) * 100

print(f"\n[RF (hourly + 증강) 성능]")
print(f"  RMSE : {rmse:.8f} kWh")
print(f"  MAE  : {mae:.8f} kWh")
print(f"  R2   : {r2:.8f}")
print(f"  MAPE : {mape:.4f}%")

# Feature Importance
importance = pd.DataFrame({
    'feature'   : features,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)
print(f"\n[Feature Importance]")
print(importance.to_string(index=False))


[RF (hourly + 증강) 성능]
  RMSE : 0.00046000 kWh
  MAE  : 0.00018298 kWh
  R2   : 0.99997937
  MAPE : 0.1389%

[Feature Importance]
 feature  importance
duration    0.998471
     cpu    0.001493
  memory    0.000035
    hour    0.000002


In [8]:
# ============================================================
# 모델 저장
# ============================================================
model_file = os.path.join(MODEL_PATH, config['model']['model_names']['rf_hourly'])
with open(model_file, 'wb') as f:
    pickle.dump(rf, f)
print(f"\nSaved: {model_file} ({os.path.getsize(model_file)/(1024**2):.1f} MB)")
print(f"feature_names_in_: {rf.feature_names_in_}")

# ============================================================
# 결과 저장
# ============================================================
result_file = os.path.join(REPORT_PATH, config['model']['results']['results_json_hourly'])

results = {
    'method'   : 'RandomForest (hourly aggregation + train augmentation)',
    'features' : features,
    'target'   : target,
    'rf_params': {
        'n_estimators'     : 100,
        'max_depth'        : 15,
        'min_samples_split': 10,
        'min_samples_leaf' : 5,
    },
    'augmentation': {
        'multipliers' : [2, 3, 4, 6, 8, 12],
        'max_duration': MAX_DURATION,
        'train_rows'  : len(X_train_aug),
    },
    'metrics': {
        'rmse': float(rmse),
        'mae' : float(mae),
        'r2'  : float(r2),
        'mape': float(mape),
    },
    'feature_importance': importance.set_index('feature')['importance'].to_dict(),
    'model_file': config['model']['model_names']['rf_hourly'],
}

with open(result_file, 'w') as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print(f"Results saved: {result_file}")
print("Done!")


Saved: /content/drive/MyDrive/EcoTracing/models/energy_model_rf_hourly.pkl (300.5 MB)
feature_names_in_: ['cpu' 'memory' 'duration' 'hour']
Results saved: /content/drive/MyDrive/EcoTracing/outputs/reports/phase1_results_hourly.json
Done!
